# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
import ollama

client = ollama.Client()

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


In [3]:
print("Total documents:", len(documents))
print("Data type:", type(documents))
print("First document:", documents[0])
print("Empty documents:", sum(not doc.strip() for doc in documents))

Total documents: 59
Data type: <class 'list'>
First document: Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.
Empty documents: 0


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan `nomic-embed-text:latest`.

In [4]:
def get_embedding(text):
    result = client.embed(
        model='nomic-embed-text:latest',  # sesuaikan versi model embedding
        input=f"search_document: {text}"
    )
    return result.embeddings[0]

def get_query_embedding(query):
    result = client.embed(
        model='nomic-embed-text:latest',
        input=f"search_query: {query}"
    )
    return result.embeddings[0]

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 768)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [5]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [6]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_query_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.49251363)}
{'rank': 2, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.51427275)}
{'rank': 3, 'question': 'Bagaimana cara melacak paket yang sedang dikirim?', 'answer': "Nomor resi akan muncul di halaman 'Pesanan Saya' setelah barang dikirim, dan dapat digunakan untuk melacak posisi paket di website kurir terkait.", 'category': 'Pengiriman', 'distance': np.float32(0.57892585)}


In [7]:
# Mencoba menjalankan Tiga Query

queries = [
    "Bagaimana cara mengembalikan barang?",
    "Saya ingin membatalkan pesanan yang sudah dibuat.",
    "Apakah platform ini menyediakan layanan perbaikan laptop?"
]

for query in queries:
    print(f"\nQuery: {query}")

    results = semantic_search(query, top_k=3)

    for result in results:
        print(
            f"Rank: {result['rank']} | "
            f"Category: {result['category']} | "
            f"Distance: {result['distance']:.4f}"
        )
        print(f"FAQ: {result['question']}")


Query: Bagaimana cara mengembalikan barang?
Rank: 1 | Category: Pengembalian | Distance: 0.5014
FAQ: Bagaimana prosedur pengembalian barang yang tidak sesuai?
Rank: 2 | Category: Pengembalian | Distance: 0.5353
FAQ: Bagaimana jika penjual menolak pengajuan pengembalian saya?
Rank: 3 | Category: Customer Service | Distance: 0.5388
FAQ: Bagaimana cara melaporkan penjual yang menipu?

Query: Saya ingin membatalkan pesanan yang sudah dibuat.
Rank: 1 | Category: Pemesanan | Distance: 0.3681
FAQ: Bagaimana cara membatalkan pesanan yang sudah dibuat?
Rank: 2 | Category: Produk | Distance: 0.5163
FAQ: Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
Rank: 3 | Category: Pemesanan | Distance: 0.5394
FAQ: Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?

Query: Apakah platform ini menyediakan layanan perbaikan laptop?
Rank: 1 | Category: Akun | Distance: 0.4704
FAQ: Apakah data pribadi saya aman di platform ini?
Rank: 2 | Category: Pemesanan | Distance: 0.4847
FA

## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [8]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. 

Aturan:
    1. Prioritaskan FAQ dengan ranking pertama sebagai sumber utama jawaban.
    2. Gunakan FAQ ranking berikutnya hanya jika diperlukan untuk melengkapi informasi.
    3. Jangan menggabungkan informasi yang tidak relevan.
    4. Jangan mengarang informasi di luar konteks FAQ.
    5. Jika tidak ada FAQ yang relevan, katakan bahwa informasi belum tersedia.


Konteks FAQ relevan:
{context}

Pertanyaan pengguna: 
{query}

Jawaban:
'''
    
    return prompt

## 7. Mengintegrasikan LLM (Local Ollama model: gpt-oss:20b)

In [15]:
def generate_answer(query, top_k=3):

    # 1. Retrieval
    search_results = semantic_search(query, top_k=top_k)

    # 2. Jika tidak ada hasil retrieval
    if not search_results:
        return "Informasi belum tersedia.", []

    # 3. Build prompt
    prompt = build_prompt(query, search_results)

    # 4. Generate answer
    response = client.chat(
        model='gpt-oss:20b',
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0.3}
    )

    return response.message.content, search_results

In [21]:
query = "Apakah saya bisa membayar dengan cicilan?"

answer, sources = generate_answer(query, top_k=3)

# ==========================================
# OUTPUT
# ==========================================

print("=" * 80)
print("SEMANTIC SEARCH & LLM APPLICATION")
print("=" * 80)

print(f"\nQUERY:\n{query}")

# ------------------------------------------
# 1. HASIL RETRIEVAL
# ------------------------------------------

print("\n" + "-" * 80)
print("1. HASIL RETRIEVAL")
print("-" * 80)

if sources:
    for source in sources:
        print(
            f"\nRank {source['rank']} | "
            f"Distance: {source['distance']:.6f}"
        )
        print(f"FAQ: {source['question']}")
else:
    print("Tidak ada hasil retrieval.")

# ------------------------------------------
# 2. JAWABAN LLM
# ------------------------------------------

print("\n" + "-" * 80)
print("2. JAWABAN LLM")
print("-" * 80)

if answer and answer.strip():
    print(answer.strip())
else:
    print("LLM tidak menghasilkan jawaban.")

# ------------------------------------------
# 3. SUMBER FAQ
# ------------------------------------------

print("\n" + "-" * 80)
print("3. SUMBER FAQ")
print("-" * 80)

if sources:
    for source in sources:
        print(f"\n[{source['rank']}] {source['question']}")
        print(f"Jawaban : {source['answer']}")
        print(f"Distance: {source['distance']:.6f}")
else:
    print("Tidak ada sumber FAQ.")

print("\n" + "=" * 80)
print("END OF RESULT")
print("=" * 80)

SEMANTIC SEARCH & LLM APPLICATION

QUERY:
Apakah saya bisa membayar dengan cicilan?

--------------------------------------------------------------------------------
1. HASIL RETRIEVAL
--------------------------------------------------------------------------------

Rank 1 | Distance: 0.480865
FAQ: Apakah saya bisa membayar dengan cicilan?

Rank 2 | Distance: 0.494862
FAQ: Apakah saya bisa memesan tanpa membuat akun?

Rank 3 | Distance: 0.543284
FAQ: Apakah biaya ongkos kirim pengembalian ditanggung pembeli?

--------------------------------------------------------------------------------
2. JAWABAN LLM
--------------------------------------------------------------------------------
Ya, Anda dapat membayar dengan cicilan. Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu, namun ada nominal minimal yang ditentukan oleh pihak bank.

--------------------------------------------------------------------------------
3. SUMBER FAQ
--------------------------------------------

## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [22]:
test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
    
    # 1. Direct query: pertanyaan yang jawabannya tersedia di FAQ
    "Bagaimana cara melakukan pemesanan di platform ini?",

    # 2. Direct query: pertanyaan yang jawabannya tersedia di FAQ
    "Berapa lama estimasi waktu pengiriman barang?",

    # 3. Paraphrase: menggunakan kalimat berbeda dari FAQ asli
    "Kalau saya ingin produk dengan model tertentu, apakah bisa request langsung ke seller?",

    # 4. Paraphrase: menggunakan kalimat berbeda dari FAQ asli
    "Seusai melakukan transfer ke rekening penjual, apakah saya perlu mengirim bukti pembayaran?",

    # 5. Out-of-Knowledge-Base: informasi tidak tersedia di FAQ
    "Siapa yang membakar hutan di Kalimantan, Papua, dan Ujung Kulon?"
]

for i, q in enumerate(test_queries, start=1):

    print("\n" + "=" * 70)
    print(f"TEST CASE {i}")
    print(f"QUERY: {q}")

    answer, sources = generate_answer(q)

    print("\nHASIL RETRIEVAL:")

    if sources:
        for s in sources:
            print(f"Rank     : {s['rank']}")
            print(f"Question : {s['question']}")
            print(f"Distance : {s['distance']:.6f}")
            print("-" * 50)
    else:
        print("Tidak ada hasil retrieval.")

    print("\nJAWABAN LLM:")
    print(answer)

    print("\nSUMBER FAQ:")
    if sources:
        for s in sources:
            print(f"- Rank {s['rank']}: {s['question']}")
    else:
        print("Tidak ada sumber FAQ.")

    print("=" * 70)


TEST CASE 1
QUERY: Bagaimana cara melakukan pemesanan di platform ini?

HASIL RETRIEVAL:
Rank     : 1
Question : Bagaimana cara melakukan pemesanan di platform ini?
Distance : 0.280459
--------------------------------------------------
Rank     : 2
Question : Bagaimana cara melacak paket yang sedang dikirim?
Distance : 0.527213
--------------------------------------------------
Rank     : 3
Question : Bagaimana cara melaporkan penjual yang menipu?
Distance : 0.555569
--------------------------------------------------

JAWABAN LLM:
Pilih produk yang diinginkan, klik **'Tambah ke Keranjang'**, lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.

SUMBER FAQ:
- Rank 1: Bagaimana cara melakukan pemesanan di platform ini?
- Rank 2: Bagaimana cara melacak paket yang sedang dikirim?
- Rank 3: Bagaimana cara melaporkan penjual yang menipu?

TEST CASE 2
QUERY: Berapa lama estimasi waktu pengiriman barang?

HASIL RETRIEVAL:
Rank     : 1
Question : Berapa lam

### Catatan Analisis Relevansi

**Analisis Hasil Pengujian dan Relevansi Retrieval**

Berdasarkan lima skenario pengujian, sistem semantic search dan LLM berhasil menjalankan pipeline end-to-end, mulai dari menerima query pengguna, melakukan retrieval menggunakan FAISS, hingga menghasilkan jawaban melalui Ollama dengan model `gpt-oss:20b`.

Pada pengujian direct query, sistem menunjukkan performa retrieval yang baik. FAQ yang relevan berhasil ditemukan pada peringkat pertama untuk kedua query yang diuji. Hal ini menunjukkan bahwa embedding mampu mengenali kesamaan antara pertanyaan pengguna dan pertanyaan dalam knowledge base.

Pada pengujian paraphrase, hasilnya menunjukkan performa yang belum konsisten. Query mengenai custom produk berhasil menemukan FAQ yang relevan pada peringkat pertama. Namun, query mengenai konfirmasi bukti pembayaran hanya menemukan FAQ yang relevan pada peringkat ketiga. Meskipun FAQ tersebut masih berada dalam Top 3, dua hasil teratas tidak memiliki relevansi langsung dengan pertanyaan pengguna.

Dari sisi LLM, seluruh query yang memiliki jawaban dalam knowledge base menghasilkan respons yang relevan. Pada pengujian out-of-knowledge-base, LLM juga berhasil menyatakan bahwa informasi belum tersedia, sehingga tidak memberikan jawaban yang tidak didukung oleh knowledge base.

Secara keseluruhan, Hit Rate @1 mencapai 75%, sedangkan Hit Rate @3 mencapai 100% pada empat query yang memiliki jawaban dalam knowledge base. Hasil ini menunjukkan bahwa sistem telah berhasil menemukan FAQ yang relevan, tetapi kualitas urutan ranking masih dapat ditingkatkan.

Keterbatasan utama yang ditemukan adalah kemampuan embedding dalam menangkap hubungan semantik pada beberapa query paraphrase. Oleh karena itu, pengembangan selanjutnya dapat dilakukan dengan membandingkan Question-only embedding, Question + Answer embedding, Cosine Similarity, serta pendekatan hybrid retrieval.

**Kesimpulan:** Sistem telah memenuhi fungsi dasar semantic search dan LLM-powered response. Namun, optimasi retrieval masih diperlukan agar FAQ yang paling relevan secara konsisten muncul pada peringkat pertama dan kualitas jawaban semakin terjaga berdasarkan konteks knowledge base.